# Medical RAG Bot - research notebook

A place to experiment with the pipeline (chunking, retrieval, prompt, model) 

In [1]:
import os

if os.path.basename(os.getcwd()) == "research":
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: d:\Downloads\RAG-Based-Medical-ChatBot-main


## 1. Environment variables (Pinecone + LangSmith)

In [2]:
from dotenv import load_dotenv

os.environ["LANGSMITH_PROJECT"] = "medical-bot-notebook"
load_dotenv()

assert os.getenv("PINECONE_API_KEY"), "PINECONE_API_KEY missing in .env"
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "medical-bot")
print("Index            :", INDEX_NAME)
print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING"), "| project:", os.getenv("LANGSMITH_PROJECT"))

Index            : medical-bot
LangSmith tracing: true | project: medical-bot-notebook


## 2. Import the same helpers the app uses


In [3]:
from src.helper import (
    load_pdf_file,
    filter_to_minimal_docs,
    text_split,
    download_hugging_face_embeddings,
    load_llm,
)

d:\Downloads\RAG-Based-Medical-ChatBot-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Inspect the data and the chunks (local only, nothing is uploaded)

In [4]:
extracted_data = load_pdf_file("data/")
minimal_docs = filter_to_minimal_docs(extracted_data)
text_chunks = text_split(minimal_docs)

print("Pages :", len(extracted_data))
print("Chunks:", len(text_chunks))

Pages : 1505
Chunks: 7308


In [5]:
from collections import Counter

for src, n in Counter(c.metadata["source"] for c in text_chunks).most_common():
    print(f"{n:5d}  {src}")

 4871  data\First Principles of Gastroenterology.pdf
 1134  data\Diabetes.pdf
  863  data\STANDARD TREATMENT.pdf
  258  data\Allergen Immunotherapy.pdf
  182  data\clinical_medicine_ashok_chandra.pdf


In [6]:
# Chunk length distribution: most chunks should be close to chunk_size=500
lengths = [len(c.page_content) for c in text_chunks]
print("min:", min(lengths), "| max:", max(lengths), "| avg:", round(sum(lengths) / len(lengths)))
print("chunks under 100 chars:", sum(l < 100 for l in lengths))

print("\n--- sample chunk ---")
print(text_chunks[100].metadata)
print(text_chunks[100].page_content)

min: 3 | max: 500 | avg: 419
chunks under 100 chars: 277

--- sample chunk ---
{'source': 'data\\Allergen Immunotherapy.pdf'}
14
Allergen Extract Treatment Set Preparation Hints
1)  Do not mix prescriptions for more than one patient at 
a time.
2)  Train multiple qualified personnel in allergen extract 
preparation in case of absences and for participation 
in quality checks.
3)  Avoid putting hand lotion on before the 
compounding of allergen extract and skin test 
antigens. Lotion tends to harbor bacteria.
4)  Regularly review operating procedures for opportunities 
to make the process safer and more efficient.


## 4. Embedding model

In [7]:
embeddings = download_hugging_face_embeddings()

vector = embeddings.embed_query("What is diabetes?")
print("Vector length:", len(vector))

Vector length: 384


## 5. Connect to the EXISTING Pinecone index (read-only)

In [8]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
print("Indexes in your account:", pc.list_indexes().names())

stats = pc.Index(INDEX_NAME).describe_index_stats()
print("Vectors in", INDEX_NAME, ":", stats.get("total_vector_count"))

Indexes in your account: ['medical-bot']
Vectors in medical-bot : 7308


In [9]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embeddings,
)

## 6. Retrieval - look at what the retriever finds before involving the LLM

In [10]:
def show_retrieval(question, k=3):
    results = docsearch.similarity_search_with_score(question, k=k)
    print(f"Q: {question}\n")
    for doc, score in results:
        print(f"score={score:.3f} | {doc.metadata['source']}")
        print("   ", doc.page_content[:200].replace("\n", " "), "...\n")

show_retrieval("What are the symptoms of diabetes?")

Q: What are the symptoms of diabetes?

score=0.550 | data\Diabetes.pdf
    groups than others.  Diabetes,  which is pretty prevalent, is an illness that may  result in many different experiences that  require support.  Experiences such as  diagnosis, the switch from  lifesty ...

score=0.540 | data\Diabetes.pdf
    3   Complications of diabetes  include the progressive  damage, dysfunction and  failure of various organs including the kidneys, nerves, heart, eyes and blood  vessels.3 Diabetes can also affect the  ...

score=0.527 | data\Diabetes.pdf
    please tell us how strongly the item describes how you feel about having diabetes. On a scale of  1: does not describe me at all 3: describes me moderately well 5: describes me very well    I believe  ...



In [11]:
show_retrieval("How should allergen extracts be stored?")
show_retrieval("Who won the 2011 Cricket World Cup?")

Q: How should allergen extracts be stored?

score=0.709 | data\Allergen Immunotherapy.pdf
    16 VIII. ALLERGEN EXTRACT STABILITY AND STORAGE The stability and potency of allergen extracts can be  compromised by elevated temperatures, contamination  and protease degradation of key allergenic p ...

score=0.685 | data\Allergen Immunotherapy.pdf
    due to its effect on proteolytic enzymes. Less is known  about the effects of freezing (<0ºC) on allergen extract  potency, but at least one study found a moderate loss of  potency when an extract was ...

score=0.683 | data\Allergen Immunotherapy.pdf
    treatment and diagnostic sets are no later than the  stock extract used with the earliest expiration date. •  Discard or separate products that have expired. •  Ensure that personal allergen extract s ...

Q: Who won the 2011 Cricket World Cup?

score=0.286 | data\Diabetes.pdf
     Achieved this goal every day in the past week ...

score=0.286 | data\Diabetes.pdf
     Achieved this goal ev

## 7. Load the LLM (Qwen via the Groq API)

In [12]:
chatModel = load_llm()
print("Model:", chatModel.model_name)

Model: openai/gpt-oss-120b


## 8. Build the RAG chain (same as app.py)

In [13]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from src.prompt import system_prompt

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

def build_rag_chain(k=3):
    retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": k})
    question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
    return create_retrieval_chain(retriever, question_answer_chain)

rag_chain = build_rag_chain(k=3)

In [14]:
import time

def ask(question, chain=None, run_name="notebook-ask"):
    chain = chain or rag_chain
    start = time.time()
    response = chain.invoke({"input": question}, config={"run_name": run_name})
    print(f"Q: {question}")
    print(f"A: {response['answer'].strip()}")
    print(f"   ({time.time() - start:.1f}s) sources:",
          sorted({d.metadata['source'] for d in response['context']}))
    print()
    return response

ask("What are the symptoms of diabetes?")
ask("How should allergen extracts be stored?")
ask("Who won the 2011 Cricket World Cup?")

Q: What are the symptoms of diabetes?
A: Common early symptoms of diabetes include excessive thirst, frequent urination, and unexplained weight loss. Other signs can be persistent fatigue, blurred vision, and slow‑healing cuts or infections. If you notice any of these, it’s important to see a healthcare provider for testing.
   (1.8s) sources: ['data\\Diabetes.pdf']

Q: How should allergen extracts be stored?
A: Allergen extracts should be kept at the manufacturer‑recommended temperature (typically refrigerated) and protected from freezing or repeated freeze‑thaw cycles. They should be stored in designated trays, kept cool during any prolonged mixing, and any expired product should be discarded. Additives and preservatives in the extracts help maintain stability when these storage conditions are followed.
   (1.1s) sources: ['data\\Allergen Immunotherapy.pdf']

Q: Who won the 2011 Cricket World Cup?
A: India won the 2011 Cricket World Cup, defeating Sri Lanka in the final.
   (1.1s) so

{'input': 'Who won the 2011 Cricket World Cup?',
 'context': [Document(id='c393c1d2-aec9-4fef-a0df-293e8494a610', metadata={'source': 'data\\Diabetes.pdf'}, page_content='\x80 Achieved this goal every day in the past week'),
  Document(id='ad702622-1b10-4b6b-8fa0-9cfdb3fe1ad6', metadata={'source': 'data\\Diabetes.pdf'}, page_content='\x80 Achieved this goal every day in the past week'),
  Document(id='f8ff991d-4575-44b3-9176-7fd3a6d477cd', metadata={'source': 'data\\First Principles of Gastroenterology.pdf'}, page_content='o East Africa \no Sweden o Canada (whites) o Japan o Canada (Inuit) \no Chile o Russia o Southeast Asia o Indonesia \no Czechoslovakia o United Kingdom o Northern India o West Africa \no United States \n(Hispanics) \no Australia  \no Italy  \no Germany \no Greece  \no Portugal \no Southern Africa')],
 'answer': 'India won the 2011 Cricket World Cup, defeating Sri\u202fLanka in the final.'}

## 9. Experiment: does retrieving more chunks change the answer?

In [15]:
rag_chain_k5 = build_rag_chain(k=5)

question = "What is the first-line treatment for type 2 diabetes?"
ask(question, rag_chain, run_name="k=3")
ask(question, rag_chain_k5, run_name="k=5")

Q: What is the first-line treatment for type 2 diabetes?
A: I’m sorry, but I don’t have that information in the provided context.
   (1.3s) sources: ['data\\Diabetes.pdf']

Q: What is the first-line treatment for type 2 diabetes?
A: I don’t know.
   (1.1s) sources: ['data\\Diabetes.pdf', 'data\\STANDARD TREATMENT.pdf']



{'input': 'What is the first-line treatment for type 2 diabetes?',
 'context': [Document(id='c0401234-958d-45d4-8e3c-8ceb241e5c94', metadata={'source': 'data\\Diabetes.pdf'}, page_content='type 2 diabetes patients from a University of Michigan ambulatory care clinic in \nBrighton, MI.  This clinic has a diabetes care prog ram that includes individual \ndisease management by a clinical pharmaci st.  Appointments usually consist of \nthe pharmacist reviewing the patient’s ch art for test results (such as HbA1c), \nmedication information, and other health ca re provider notes.  The patient may \nalso bring their medication taking and blood glucose monito ring logs, which the'),
  Document(id='18a603c5-13f6-4e7d-bff4-8c95cb323240', metadata={'source': 'data\\Diabetes.pdf'}, page_content='Would you consider joining an online support group for type 2 diabetes? \n \n\x80 Yes  \n\x80 No'),
  Document(id='cb6b1be1-965d-43b5-839b-988b5cb6ef98', metadata={'source': 'data\\Diabetes.pdf'}, page_con